In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# TODO: Fill in the Google Drive path where you uploaded the project
# Example: If you create a 2020FA folder and put all the files under A1 folder, then '2020FA/A1'
# GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = '2020FA/A1'
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Exp5'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

['AW-SSJF_80jobs_load0.50_fixed_jobs.csv', 'AW-SSJF_80jobs_load0.70_fixed_jobs.csv', 'AW-SSJF_80jobs_load0.90_fixed_jobs.csv', 'Untitled0.ipynb']


In [3]:
%cd /content/drive/MyDrive/Exp5

/content/drive/MyDrive/Exp5


In [4]:
!ls

AW-SSJF_80jobs_load0.50_fixed_jobs.csv	AW-SSJF_80jobs_load0.90_fixed_jobs.csv
AW-SSJF_80jobs_load0.70_fixed_jobs.csv	Untitled0.ipynb


In [7]:
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# -----------------------
# 0) Find data in CURRENT folder
# -----------------------
HERE = Path(".")
csv_files = sorted([p for p in HERE.rglob("*.csv") if p.is_file()])
assert len(csv_files) > 0, "No CSV files found in the current folder."

print("Found CSV files:")
for p in csv_files:
    print(" -", p.name)


# -----------------------
# 1) Helpers
# -----------------------
def parse_load_from_filename(name: str):
    m = re.search(r"load(\d+(?:\.\d+)?)", name)
    return float(m.group(1)) if m else None

def normalize_native(df: pd.DataFrame) -> pd.DataFrame:
    # map your native columns -> canonical
    rename_map = {
        "waiting_time": "wait_time",
        "actual_serving_time": "service_time",
        "predicted_serving_time": "predicted_S",
        "end_time": "finish_time",
        "completion_time": "finish_time",
    }
    for k, v in rename_map.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k: v})

    # derive wait/service if missing
    if "wait_time" not in df.columns and {"arrival_time","start_time"}.issubset(df.columns):
        df["wait_time"] = df["start_time"] - df["arrival_time"]
    if "service_time" not in df.columns and {"start_time","finish_time"}.issubset(df.columns):
        df["service_time"] = df["finish_time"] - df["start_time"]

    return df

def compute_queue_length_fast(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute at each job arrival:
      queue_length (in system, excl self):  arrived - finished - 1
      queue_waiting_length (waiting only):  arrived - started  - 1
    Uses searchsorted => O(n log n).
    Requires arrival_time/start_time/finish_time.
    """
    need = {"arrival_time","start_time","finish_time"}
    if not need.issubset(df.columns):
        return df

    d = df.sort_values("arrival_time").reset_index(drop=True).copy()
    arr = d["arrival_time"].to_numpy(float)
    st  = d["start_time"].to_numpy(float)
    fin = d["finish_time"].to_numpy(float)

    arr_sorted = np.sort(arr)
    st_sorted  = np.sort(st)
    fin_sorted = np.sort(fin)

    arrived  = np.searchsorted(arr_sorted, arr, side="right")
    started  = np.searchsorted(st_sorted,  arr, side="right")
    finished = np.searchsorted(fin_sorted, arr, side="right")

    in_system = arrived - finished - 1
    waiting   = arrived - started  - 1

    d["queue_length"] = np.maximum(in_system, 0).astype(int)
    d["queue_waiting_length"] = np.maximum(waiting, 0).astype(int)
    return d

def mg1_Wq(lam, ES, ES2):
    rho = lam * ES
    if rho >= 1:
        return np.inf
    return (lam * ES2) / (2*(1-rho))

def estimate_lambda(g: pd.DataFrame) -> float:
    # lambda ~= N / (max(finish)-min(arrival))
    t0 = float(g["arrival_time"].min())
    t1 = float(g["finish_time"].max())
    return float(len(g) / max(1e-9, (t1 - t0)))


# -----------------------
# 2) Load all CSVs (from current folder) + normalize
# -----------------------
parts = []
for fp in csv_files:
    df = pd.read_csv(fp)
    df["__source_file__"] = fp.name
    rho = parse_load_from_filename(fp.name)
    if rho is not None:
        df["target_rho"] = rho
    df = normalize_native(df)
    # compute queue_length if missing
    if "queue_length" not in df.columns and {"arrival_time","start_time","finish_time"}.issubset(df.columns):
        df = compute_queue_length_fast(df)
    parts.append(df)

jobs = pd.concat(parts, ignore_index=True)

need_cols = ["arrival_time","start_time","finish_time","wait_time","service_time"]
missing = [c for c in need_cols if c not in jobs.columns]
assert not missing, f"Missing required columns after normalization: {missing}"

print("\nCombined shape:", jobs.shape)
print("Columns:", list(jobs.columns))


# -----------------------
# 3) Summarize per load and validate M/G/1
# -----------------------
load_key = "target_rho" if "target_rho" in jobs.columns else "__source_file__"

rows = []
for lv, g in jobs.groupby(load_key):
    g = g.replace([np.inf, -np.inf], np.nan).dropna(subset=["wait_time","service_time"]).copy()
    if len(g) < 5:
        continue

    ES  = float(g["service_time"].mean())
    ES2 = float((g["service_time"]**2).mean())
    empWq = float(g["wait_time"].mean())
    lam = estimate_lambda(g)
    rho_est = lam * ES
    predWq = mg1_Wq(lam, ES, ES2)

    rows.append({
        "load": float(lv) if isinstance(lv, (int,float,np.number)) else str(lv),
        "n_jobs": int(len(g)),
        "lambda_est": lam,
        "E_S": ES,
        "E_S2": ES2,
        "rho_est": rho_est,
        "emp_mean_Wq": empWq,
        "mg1_pred_mean_Wq": predWq,
        "residual_emp_minus_pred": (empWq - predWq) if np.isfinite(predWq) else np.nan
    })

summary = pd.DataFrame(rows).sort_values("rho_est")
summary


# -----------------------
# 4) Outputs
# -----------------------
outdir = Path("./exp5_out")
outdir.mkdir(exist_ok=True, parents=True)

summary.to_csv(outdir / "exp5_summary.csv", index=False)

# 4.1 mg1_validation.png
finite = summary.replace([np.inf,-np.inf], np.nan).dropna(subset=["emp_mean_Wq","mg1_pred_mean_Wq"]).copy()
plt.figure()
plt.scatter(finite["emp_mean_Wq"], finite["mg1_pred_mean_Wq"], s=30)
lo = float(min(finite["emp_mean_Wq"].min(), finite["mg1_pred_mean_Wq"].min()))
hi = float(max(finite["emp_mean_Wq"].max(), finite["mg1_pred_mean_Wq"].max()))
plt.plot([lo,hi],[lo,hi])
plt.xlabel("Empirical mean Wq")
plt.ylabel("M/G/1 predicted mean Wq")
plt.title("M/G/1 validation (mean waiting in queue)")
plt.tight_layout()
plt.savefig(outdir / "mg1_validation.png", dpi=200)
plt.close()

# 4.2 residual_vs_load.png
finite_res = summary.dropna(subset=["residual_emp_minus_pred"]).copy()
plt.figure()
plt.scatter(finite_res["rho_est"], finite_res["residual_emp_minus_pred"], s=30)
plt.axhline(0.0)
plt.xlabel("rho_est")
plt.ylabel("Residual (empirical - predicted)")
plt.title("Residual vs utilization")
plt.tight_layout()
plt.savefig(outdir / "residual_vs_load.png", dpi=200)
plt.close()


# -----------------------
# 5) Regression: predict wait_time from queue_length and predicted_S (or service_time)
# -----------------------
scol = "predicted_S" if "predicted_S" in jobs.columns else "service_time"
assert "queue_length" in jobs.columns, "queue_length not found (could not compute)."

reg_df = jobs[["queue_length", scol, "wait_time"]].replace([np.inf,-np.inf], np.nan).dropna().copy()
X = pd.DataFrame({
    "queue_length": reg_df["queue_length"].astype(float),
    "predicted_S": reg_df[scol].astype(float),
})
X["qlen_x_S"] = X["queue_length"] * X["predicted_S"]
y = reg_df["wait_time"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", Ridge(alpha=1.0))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = float(mean_absolute_error(y_test, y_pred))
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
r2 = float(r2_score(y_test, y_pred))

plt.figure()
plt.scatter(y_test, y_pred, s=10)
lo = float(min(y_test.min(), y_pred.min()))
hi = float(max(y_test.max(), y_pred.max()))
plt.plot([lo,hi],[lo,hi])
plt.xlabel("Actual wait_time")
plt.ylabel("Predicted wait_time")
plt.title("Regression: predicted vs actual (test set)")
plt.tight_layout()
plt.savefig(outdir / "regression_scatter.png", dpi=200)
plt.close()

report = {
    "data_files": [p.name for p in csv_files],
    "n_rows_total": int(len(jobs)),
    "mg1_metrics_finite_pred": {
        "n_points": int(len(finite)),
        "MAE": float(np.mean(np.abs(finite["emp_mean_Wq"] - finite["mg1_pred_mean_Wq"]))) if len(finite) else None,
        "RMSE": float(np.sqrt(np.mean((finite["emp_mean_Wq"] - finite["mg1_pred_mean_Wq"])**2))) if len(finite) else None,
        "note": "Points with rho>=1 lead to infinite MG1 mean; those are excluded from finite metrics."
    },
    "regression": {
        "features": ["queue_length", "predicted_S", "qlen_x_S"],
        "used_S_column": scol,
        "model": "Ridge(alpha=1.0) + StandardScaler",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "coef_scaled_space": {
            "intercept": float(model.named_steps["reg"].intercept_),
            "coef": model.named_steps["reg"].coef_.tolist()
        }
    }
}

with open(outdir / "exp5_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\n[OK] Wrote outputs to:", outdir.resolve())
print(" -", (outdir / "mg1_validation.png").name)
print(" -", (outdir / "residual_vs_load.png").name)
print(" -", (outdir / "regression_scatter.png").name)
print(" -", (outdir / "exp5_summary.csv").name)
print(" -", (outdir / "exp5_report.json").name)


Found CSV files:
 - AW-SSJF_80jobs_load0.50_fixed_jobs.csv
 - AW-SSJF_80jobs_load0.70_fixed_jobs.csv
 - AW-SSJF_80jobs_load0.90_fixed_jobs.csv

Combined shape: (240, 26)
Columns: ['job_id', 'emotion_label', 'arousal', 'valence', 'russell_quadrant', 'affect_weight', 'urgency', 'emotion_confidence', 'arrival_time', 'predicted_S', 'service_time', 'start_time', 'finish_time', 'wait_time', 'turnaround_time', 'response_text', 'output_token_length', 'cached', 'error_msg', 'fallback_used', 'model_name', 'conversation_context_preview', '__source_file__', 'target_rho', 'queue_length', 'queue_waiting_length']

[OK] Wrote outputs to: /content/drive/MyDrive/Exp5/exp5_out
 - mg1_validation.png
 - residual_vs_load.png
 - regression_scatter.png
 - exp5_summary.csv
 - exp5_report.json
